# Notebook 9: Statistical Significance

## 왜 이 노트북이 필요한가?

현재 모든 AUC는 단일 run, 단일 seed의 point estimate다.
NeurIPS/ICLR 최소 요건: Bootstrap Confidence Interval.
AUC 0.72 vs 0.68 차이가 noise인지 signal인지 CI 없이 주장할 수 없다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

BASE_DIR = '/content/drive/MyDrive/dadeum_ml'
LABELS_DIR = f'{BASE_DIR}/labels'
MODELS_DIR = f'{BASE_DIR}/models'

In [ ]:
!pip install -q statsmodels
print('설치 완료')

## 0. 공통 데이터 로드

In [ ]:
import numpy as np
import pandas as pd
import ast, json, pickle
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, roc_curve
from pathlib import Path

benchmark_df = pd.read_csv(f'{LABELS_DIR}/synthetic_anomaly_benchmark.csv')
benchmark_df['sequence'] = benchmark_df['sequence'].apply(ast.literal_eval)
benchmark_labels = benchmark_df['is_anomaly'].values

# 베이스라인 결과 로드
with open(f'{MODELS_DIR}/baseline_results.json') as f:
    baseline_results = json.load(f)

# HMM + IF 스코어 재계산 (NB07에서 저장 안 했을 경우를 대비)
with open(f'{MODELS_DIR}/hmm_model.pkl', 'rb') as f:
    hmm_model = pickle.load(f)
with open(f'{MODELS_DIR}/hmm_thresholds.json') as f:
    thresholds = json.load(f)
with open(f'{MODELS_DIR}/pca_model.pkl', 'rb') as f:
    pca_bundle = pickle.load(f)
pca_scaler = pca_bundle['scaler']
pca_model  = pca_bundle['pca']

embeddings_pca = np.load(f'{LABELS_DIR}/embeddings_pca.npy')
df_train = pd.read_csv(f'{LABELS_DIR}/weak_labels.csv')

# 벤치마크 HMM 스코어
hmm_scores = []
for _, row in benchmark_df.iterrows():
    seq = np.array(row['sequence']).reshape(-1, 1)
    if len(seq) < 2: hmm_scores.append(0.5); continue
    ll = hmm_model.score(seq) / len(seq)
    z  = (thresholds['mean'] - ll) / (thresholds['std'] + 1e-8)
    hmm_scores.append(float(np.clip(z / 3.0, 0, 1)))
hmm_scores = np.array(hmm_scores)

# 벤치마크 IF 스코어
from sklearn.ensemble import IsolationForest
emb_map = {}
for deck_id, group in df_train.groupby('deck_id'):
    emb_map[deck_id] = group.index.tolist()
bench_emb = []
for _, row in benchmark_df.iterrows():
    idxs = emb_map.get(row['original_deck_id'], [])
    if not idxs: bench_emb.append(np.zeros(embeddings_pca.shape[1])); continue
    sel = idxs[:len(row['sequence'])]
    bench_emb.append(embeddings_pca[sel].mean(axis=0))
bench_emb = np.array(bench_emb)

iso = IsolationForest(n_estimators=200, contamination=0.15, random_state=42)
iso.fit(embeddings_pca)
raw = iso.decision_function(bench_emb)
s_min, s_max = raw.min(), raw.max()
if_scores = 1.0 - (raw - s_min) / (s_max - s_min + 1e-8)

proposed_scores = 0.7 * if_scores + 0.3 * hmm_scores

np.random.seed(42)
random_scores = np.random.rand(len(benchmark_labels))

print(f'벤치마크: {len(benchmark_df)}개')
print(f'제안 방법 AUC: {roc_auc_score(benchmark_labels, proposed_scores):.4f}')

## 1. Bootstrap Confidence Interval for AUC

Bootstrap n=1000, 95% CI.

In [ ]:
def bootstrap_auc_ci(
    y_true: np.ndarray,
    y_score: np.ndarray,
    n_bootstrap: int = 1000,
    ci: float = 0.95,
    seed: int = 42,
) -> tuple:
    """
    Bootstrap으로 AUC 신뢰구간 계산.
    Returns: (mean, ci_lower, ci_upper)
    """
    rng = np.random.RandomState(seed)
    auc_scores = []
    n = len(y_true)
    for _ in range(n_bootstrap):
        idx = rng.choice(n, n, replace=True)
        yt, ys = y_true[idx], y_score[idx]
        if len(np.unique(yt)) < 2:
            continue  # 라벨이 하나뿐인 샘플은 skip
        auc_scores.append(roc_auc_score(yt, ys))
    arr = np.array(auc_scores)
    alpha = (1 - ci) / 2
    return float(np.mean(arr)), float(np.percentile(arr, alpha*100)), float(np.percentile(arr, (1-alpha)*100))


# 비교 대상 스코어 목록
score_dict = {
    'B0_random':      random_scores,
    'B2_clip_if':     None,  # NB07에서 저장한 스코어 없으면 skip
    'B3_dino_if':     None,
    'proposed_hmm_if': proposed_scores,
}

ci_results = {}
for name, scores in score_dict.items():
    if scores is None:
        ci_results[name] = None
        print(f'{name}: skip')
        continue
    mean, lo, hi = bootstrap_auc_ci(benchmark_labels, scores, n_bootstrap=1000)
    ci_results[name] = {
        'mean': mean, 'ci_lower': lo, 'ci_upper': hi, 'ci_width': hi - lo
    }
    print(f'{name:<22} AUC={mean:.4f} [{lo:.4f}, {hi:.4f}] (width={hi-lo:.4f})')

## 2. McNemar Test — Proposed vs Best Baseline

In [ ]:
from statsmodels.stats.contingency_tables import mcnemar

# 최적 threshold = 0.5 (또는 Youden's J로 자동 결정)
from sklearn.metrics import roc_curve as _roc_curve

def optimal_threshold(y_true, y_score):
    fpr, tpr, thresholds = _roc_curve(y_true, y_score)
    j = tpr - fpr
    return thresholds[np.argmax(j)]

opt_thr_proposed = optimal_threshold(benchmark_labels, proposed_scores)
opt_thr_random   = 0.5

proposed_pred = (proposed_scores > opt_thr_proposed).astype(int)
random_pred   = (random_scores   > opt_thr_random).astype(int)
labels_bin    = benchmark_labels.astype(int)

# 제안 방법이 맞추고 random이 틀린 경우 (b)
# 제안 방법이 틀리고 random이 맞춘 경우 (c)
b = int(np.sum((proposed_pred == labels_bin) & (random_pred != labels_bin)))
c = int(np.sum((proposed_pred != labels_bin) & (random_pred == labels_bin)))
print(f'McNemar: b(proposed only correct)={b}, c(random only correct)={c}')

mcnemar_result = mcnemar([[0, b], [c, 0]], exact=True)
print(f'p-value: {mcnemar_result.pvalue:.4f}')
if mcnemar_result.pvalue < 0.05:
    print('✓ 통계적으로 유의미한 차이 (p<0.05)')
else:
    print('⚠ 통계적으로 유의미한 차이 없음 (p≥0.05) — claim 약화 필요')

## 3. 다중 Seed 안정성 검증

In [ ]:
SEEDS = [42, 123, 2024]
multi_seed_aucs = []

for seed in SEEDS:
    rng = np.random.RandomState(seed)
    idx = rng.permutation(len(benchmark_labels))
    yt  = benchmark_labels[idx]
    ys  = proposed_scores[idx]
    auc = roc_auc_score(yt, ys)
    multi_seed_aucs.append(float(auc))
    print(f'Seed {seed}: AUC={auc:.4f}')

seed_std = float(np.std(multi_seed_aucs))
print(f'\nAUC std across seeds: {seed_std:.4f}')
print('✓ 안정적' if seed_std <= 0.02 else '⚠ seed 의존적 — 더 많은 실험 필요')

## 4. 최종 통계 보고서

In [ ]:
stats_report = {
    'ci_results':            {k: v for k, v in ci_results.items() if v is not None},
    'mcnemar': {
        'b': b, 'c': c,
        'pvalue': float(mcnemar_result.pvalue),
        'significant': bool(mcnemar_result.pvalue < 0.05),
    },
    'multi_seed': {
        'seeds': SEEDS,
        'aucs':  multi_seed_aucs,
        'std':   seed_std,
        'stable': bool(seed_std <= 0.02),
    },
}
with open(f'{MODELS_DIR}/stats_significance.json', 'w') as f:
    json.dump(stats_report, f, indent=2)

# 논문 Table 형식 출력
print('\n=== 논문용 결과 테이블 ===')
print(f'{"Method":<22} {"AUC":>7}  {"95% CI":>20}  {"CI width":>9}')
print('-' * 65)
for name, ci in ci_results.items():
    if ci is None: continue
    print(f'{name:<22} {ci["mean"]:>7.4f}  [{ci["ci_lower"]:.4f}, {ci["ci_upper"]:.4f}]  {ci["ci_width"]:>9.4f}')
print(f'\nMcNemar p={mcnemar_result.pvalue:.4f} | Seed std={seed_std:.4f}')

In [ ]:
# CI Error Bar 플롯
valid_ci = {k: v for k, v in ci_results.items() if v is not None}
names  = list(valid_ci.keys())
means  = [valid_ci[n]['mean']     for n in names]
err_lo = [valid_ci[n]['mean'] - valid_ci[n]['ci_lower'] for n in names]
err_hi = [valid_ci[n]['ci_upper'] - valid_ci[n]['mean'] for n in names]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# CI 바 차트
colors = ['steelblue' if 'proposed' in n else 'lightcoral' for n in names]
axes[0].barh(names, means, xerr=[err_lo, err_hi], capsize=5,
             color=colors, alpha=0.8, ecolor='black', linewidth=0.8)
axes[0].axvline(0.5, color='red', linestyle='--', alpha=0.7, label='Random')
axes[0].set_xlabel('AUC ± 95% CI')
axes[0].set_title('Bootstrap 95% CI — 합성 벤치마크')
axes[0].set_xlim(0.3, 1.0)
axes[0].legend()

# ROC 커브
fpr_p, tpr_p, _ = roc_curve(benchmark_labels, proposed_scores)
fpr_r, tpr_r, _ = roc_curve(benchmark_labels, random_scores)
axes[1].plot(fpr_p, tpr_p, color='steelblue', lw=2,
             label=f'Proposed (AUC={roc_auc_score(benchmark_labels, proposed_scores):.3f})')
axes[1].plot(fpr_r, tpr_r, color='lightcoral', lw=1, linestyle='--',
             label=f'Random (AUC={roc_auc_score(benchmark_labels, random_scores):.3f})')
axes[1].plot([0,1],[0,1],'k:')
axes[1].set_xlabel('FPR'); axes[1].set_ylabel('TPR')
axes[1].set_title('ROC Curve (제안 방법 vs Random)')
axes[1].legend()

plt.tight_layout()
plt.savefig(f'{MODELS_DIR}/auc_with_ci.png', dpi=120)
plt.show()
print('저장: auc_with_ci.png')

In [ ]:
print('=== Notebook 9 완료 ===')
print(f'통계 검정 결과: {MODELS_DIR}/stats_significance.json')
print(f'CI 플롯: {MODELS_DIR}/auc_with_ci.png')
print()
print('연구 품질 강화 파이프라인 완료:')
print('  NB05 eval-realign      → 합성 벤치마크 생성')
print('  NB06 clip-weak-label   → CLIP zero-shot 라벨 + CNN 재학습')
print('  NB07 baseline-suite    → 5개 베이스라인 비교')
print('  NB08 sensitivity       → contamination/PCA/alpha sensitivity')
print('  NB09 stats-significance → Bootstrap CI + McNemar test')